# 02. CVAE v4 - conditional prior + residual decoder + uncertainty weight

입력은 `01_residual_cloud_fill_dnn.ipynb`가 만든 `cloud_fill_residual_v1/filled_samples`이다.

attempt3와 다른 점:

- unconditional prior `N(0,I)` 대신 `p(z | condition, meta)` conditional prior 사용
- deterministic base U-Net과 latent residual decoder를 분리
- `output = base + residual`
- posterior는 `q(z | condition, target, meta)`
- KL은 `KL(q || p)`
- `cloud_fill_uncertainty`/`target_weight`가 있으면 CVAE loss weight에 반영
- meta(`day_sin`, `day_cos`, 태양각, 위경도)는 latent가 아니라 condition/global context로 주입


In [ ]:
# 필요하면 먼저 실행
%pip install -q torch numpy pandas tqdm

from __future__ import annotations

from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


In [ ]:
# Colab runtime setup for CVAE cache training
# 전제: 02_1_build_cvae_tensor_cache.ipynb가 Drive에 cvae_tensor_cache_v4/*.pt를 만들어둔다.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def find_cache_dir(drive_root):
    candidates = [
        drive_root / 'filled_samples' / 'cvae_tensor_cache_v4',
        drive_root / 'attempt4' / 'cloud_fill_residual_v1' / 'cvae_tensor_cache_v4',
        drive_root / 'cloud_fill_residual_v1' / 'cvae_tensor_cache_v4',
    ]
    for cand in candidates:
        count = len(list(cand.glob('cvae_cache_*.pt'))) if cand.exists() else 0
        print('tensor cache candidate:', cand, 'count=', count)
        if count >= 2:
            return cand.resolve(), count
    raise FileNotFoundError('cvae tensor cache not found. Run 02_1_build_cvae_tensor_cache.ipynb first.')

def stage_cache_to_local(src, dst=Path('/content/cvae_tensor_cache_v4')):
    dst = Path(dst)
    expected = sorted(src.glob('cvae_cache_*.pt'))
    if not expected:
        raise FileNotFoundError(f'no cache pt files in {src}')
    existing_ok = dst.exists() and all((dst / p.name).exists() and (dst / p.name).stat().st_size == p.stat().st_size for p in expected)
    if existing_ok:
        print('local tensor cache already staged:', dst)
        return dst.resolve()
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    print('copy Drive tensor cache -> local SSD')
    print('src:', src)
    print('dst:', dst)
    result = subprocess.run(['bash', '-lc', f'rsync -a --info=progress2 "{src}/" "{dst}/"'], text=True)
    if result.returncode != 0:
        print('rsync failed; fallback to shutil.copytree')
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    return dst.resolve()

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    drive_cache_dir, cache_count = find_cache_dir(DRIVE_ROOT)
    local_cache_dir = stage_cache_to_local(drive_cache_dir)
    os.environ['DRIVE_CACHE_DIR'] = str(drive_cache_dir)
    os.environ['LOCAL_CACHE_DIR'] = str(local_cache_dir)
    os.environ.setdefault('ATTEMPT4_RUN_DIR', '/content/attempt4_runs')
    print('Colab CVAE cache mode')
    print('drive cache:', drive_cache_dir)
    print('local cache:', local_cache_dir)
    print('ATTEMPT4_RUN_DIR:', os.environ['ATTEMPT4_RUN_DIR'])
else:
    print('Local kernel mode')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print('CUDA:', torch.cuda.get_device_name(0))
else:
    print('CUDA not available; runtime is CPU')


In [ ]:
SEED = 20260615
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 100
STAGE1_BASE_EPOCHS = 8
PATIENCE = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
LATENT_DIM = 96
WIDTH = 32
BETA = 3.16e-5
KL_WARMUP_EPOCHS = 20
SAVE_EVERY_EPOCH = True
BALANCED_MIN_KL = 0.003

LAMBDA_GRAD = 0.10
LAMBDA_LAP = 0.03

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if os.environ.get('LOCAL_CACHE_DIR'):
    CACHE_DIR = Path(os.environ['LOCAL_CACHE_DIR']).expanduser()
elif os.environ.get('DRIVE_CACHE_DIR'):
    CACHE_DIR = Path(os.environ['DRIVE_CACHE_DIR']).expanduser()
else:
    cwd = Path.cwd()
    candidates = [
        cwd / 'cvae_tensor_cache_v4',
        cwd / 'attempt4' / 'cloud_fill_residual_v1' / 'cvae_tensor_cache_v4',
        Path('/content/cvae_tensor_cache_v4'),
        Path('/content/drive/MyDrive/attempt4/cloud_fill_residual_v1/cvae_tensor_cache_v4'),
    ]
    CACHE_DIR = next((c for c in candidates if c.exists()), candidates[-1])

if os.environ.get('ATTEMPT4_RUN_DIR'):
    RUN_BASE_DIR = Path(os.environ['ATTEMPT4_RUN_DIR']).expanduser()
else:
    RUN_BASE_DIR = Path('/content/attempt4_runs') if str(Path.cwd()).startswith('/content') else CACHE_DIR.parent
RUN_DIR = RUN_BASE_DIR / 'runs' / 'lst_cvae_condprior_residual_v4'
RUN_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = CACHE_DIR.parent
DATA_DIR = PROJECT_DIR

print('CACHE_DIR:', CACHE_DIR)
print('RUN_DIR:', RUN_DIR)
if not (CACHE_DIR / 'cvae_cache_train.pt').exists():
    raise FileNotFoundError('cvae_cache_train.pt missing. Run 02_1_build_cvae_tensor_cache.ipynb first.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR optional:', DATA_DIR)
print('CACHE_DIR:', CACHE_DIR)
print('RUN_DIR:', RUN_DIR)
print('DEVICE:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
def load_cache(split):
    path = CACHE_DIR / f'cvae_cache_{split}.pt'
    if not path.exists():
        if split == 'test':
            return None
        raise FileNotFoundError(path)
    cache = torch.load(path, map_location='cpu', weights_only=False)
    print(split, 'cache loaded:', path, 'n=', len(cache['files']))
    return cache

train_cache = load_cache('train')
val_cache = load_cache('val')
test_cache = load_cache('test')

stats = train_cache['stats']
CONDITION_CHANNELS = [str(x) for x in train_cache['condition_channels']]
TARGET_CHANNELS = [str(x) for x in train_cache['target_channels']]
ALL_TARGET_CHANNELS = [str(x) for x in train_cache['source_target_channels']]
TARGET_INDICES = [int(x) for x in train_cache['target_indices']]
META_CHANNELS = [str(x) for x in train_cache['meta_channels']]
expected_meta = ['day_sin', 'day_cos', 'sun_azimuth_norm', 'sun_elevation_norm', 'lat_norm', 'lon_norm']
if META_CHANNELS != expected_meta:
    raise RuntimeError(f'meta channel mismatch: {META_CHANNELS}')

banned = ['air', 'pm10', 'pm25', 'o3', 'pressure', 'press', 'pa', 'ndvi', 'nearest_lst']
bad = [c for c in CONDITION_CHANNELS + ALL_TARGET_CHANNELS if any(t in c.lower() for t in banned)]
if bad:
    raise RuntimeError(f'banned channels found: {bad}')

train_paths = [Path(x) for x in train_cache['files']]
val_paths = [Path(x) for x in val_cache['files']] if val_cache is not None else []
test_paths = [Path(x) for x in test_cache['files']] if test_cache is not None else []
all_paths = train_paths + val_paths + test_paths
print('samples train/val/test:', len(train_paths), len(val_paths), len(test_paths), 'total:', len(all_paths))
print('condition channels:', len(CONDITION_CHANNELS), CONDITION_CHANNELS)
print('target channels:', ALL_TARGET_CHANNELS, 'selected:', TARGET_CHANNELS)
print('meta channels:', META_CHANNELS)


In [ ]:
def as_2d_mask(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        return arr[0].astype(bool)
    return arr.astype(bool)

def cache_summary(cache):
    if cache is None:
        return None
    return {
        'n': len(cache['files']),
        'condition': tuple(cache['condition'].shape),
        'target': tuple(cache['target'].shape),
        'target_mask': tuple(cache['target_mask'].shape),
        'target_weight': tuple(cache['target_weight'].shape),
        'meta': tuple(cache['meta'].shape),
        'dtype_condition': str(cache['condition'].dtype),
        'dtype_target': str(cache['target'].dtype),
    }

print('train cache:', cache_summary(train_cache))
print('val cache:', cache_summary(val_cache))
print('test cache:', cache_summary(test_cache))


In [ ]:
def denormalize(values, channel_stats, names):
    out = values.astype(np.float32, copy=True)
    for i, name in enumerate(names):
        lo = channel_stats[name]['min']; hi = channel_stats[name]['max']
        out[i] = out[i] * (hi - lo) + lo
    return out

class CacheDataset(Dataset):
    def __init__(self, cache):
        self.cache = cache
    def __len__(self):
        return len(self.cache['files'])
    def __getitem__(self, idx):
        return {
            'condition': self.cache['condition'][idx].float(),
            'target': self.cache['target'][idx].float(),
            'target_mask': self.cache['target_mask'][idx].float(),
            'target_weight': self.cache['target_weight'][idx].float(),
            'clear_mask': self.cache['clear_mask'][idx].float(),
            'cloud_fill_mask': self.cache['cloud_fill_mask'][idx].float(),
            'meta': self.cache['meta'][idx].float(),
            'path_index': torch.tensor(idx, dtype=torch.long),
        }

train_loader = DataLoader(CacheDataset(train_cache), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
val_loader = DataLoader(CacheDataset(val_cache), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')) if val_cache is not None else None
test_loader = DataLoader(CacheDataset(test_cache), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')) if test_cache is not None else None


In [ ]:
# ### MODIFIED: MULTI-LATENT CVAE v4.1
# 단일 z가 아니라 z_global / z_season / z_weather / z_land / z_residual 로 latent block을 분리한다.
# cache는 그대로 쓰고 모델/손실/로그만 multi-latent 기준으로 바꾼다.

LATENT_BLOCK_DIMS = {
    'global': 24,
    'season': 16,
    'weather': 16,
    'land': 24,
    'residual': 16,
}
assert sum(LATENT_BLOCK_DIMS.values()) == LATENT_DIM, (LATENT_BLOCK_DIMS, LATENT_DIM)


def channel_indices(names, predicates):
    out = []
    for i, name in enumerate(names):
        low = name.lower()
        if any(pred(low) for pred in predicates):
            out.append(i)
    return out

WEATHER_IDX = channel_indices(CONDITION_CHANNELS, [
    lambda s: s in {'avg_temp', 'humidity', 'wind_u', 'wind_v', 'rain_1h'},
])
LAND_IDX = channel_indices(CONDITION_CHANNELS, [
    lambda s: s == 'albedo',
    lambda s: s.startswith('egis_cat_'),
])
STATIC_IDX = channel_indices(CONDITION_CHANNELS, [
    lambda s: s in {'elevation', 'slope', 'aspect_sin', 'aspect_cos', 'hillshade_landsat'},
])
META_INDEX = {name: i for i, name in enumerate(META_CHANNELS)}
SEASON_META_IDX = [META_INDEX[x] for x in ['day_sin', 'day_cos', 'sun_azimuth_norm', 'sun_elevation_norm'] if x in META_INDEX]
GLOBAL_META_IDX = [META_INDEX[x] for x in ['lat_norm', 'lon_norm'] if x in META_INDEX]
print('latent blocks:', LATENT_BLOCK_DIMS)
print('static idx:', [CONDITION_CHANNELS[i] for i in STATIC_IDX])
print('weather idx:', [CONDITION_CHANNELS[i] for i in WEATHER_IDX])
print('land idx:', [CONDITION_CHANNELS[i] for i in LAND_IDX[:8]], '... n=', len(LAND_IDX))
print('season meta:', [META_CHANNELS[i] for i in SEASON_META_IDX])
print('global meta:', [META_CHANNELS[i] for i in GLOBAL_META_IDX])


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1), nn.GroupNorm(min(8, out_ch), out_ch), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class FiLM(nn.Module):
    def __init__(self, context_dim, ch):
        super().__init__()
        self.to_gamma_beta = nn.Linear(context_dim, ch * 2)
    def forward(self, x, context):
        gb = self.to_gamma_beta(context)
        gamma, beta = gb.chunk(2, dim=1)
        return x * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]

class ConditionEncoder(nn.Module):
    def __init__(self, in_ch, width):
        super().__init__()
        self.s0 = ConvBlock(in_ch, width)
        self.d1 = Down(width, width)
        self.d2 = Down(width, width * 2)
        self.d3 = Down(width * 2, width * 4)
        self.d4 = Down(width * 4, width * 8)
        self.d5 = Down(width * 8, width * 8)
    def forward(self, x):
        s0 = self.s0(x)
        s1 = self.d1(s0)
        s2 = self.d2(s1)
        s3 = self.d3(s2)
        s4 = self.d4(s3)
        b = self.d5(s4)
        return b, [s0, s1, s2, s3, s4]

class Decoder(nn.Module):
    def __init__(self, bottleneck_ch, out_ch, width, context_dim, latent_ch=0, gated_skips=True):
        super().__init__()
        self.gated_skips = gated_skips
        in_ch = bottleneck_ch + latent_ch
        self.u4 = Up(in_ch, width * 8); self.f4 = ConvBlock(width * 16, width * 8); self.film4 = FiLM(context_dim, width * 8)
        self.u3 = Up(width * 8, width * 4); self.f3 = ConvBlock(width * 8, width * 4); self.film3 = FiLM(context_dim, width * 4)
        self.u2 = Up(width * 4, width * 2); self.f2 = ConvBlock(width * 4, width * 2); self.film2 = FiLM(context_dim, width * 2)
        self.u1 = Up(width * 2, width); self.f1 = ConvBlock(width * 2, width); self.film1 = FiLM(context_dim, width)
        self.u0 = Up(width, width); self.f0 = ConvBlock(width * 2, width); self.film0 = FiLM(context_dim, width)
        self.out = nn.Conv2d(width, out_ch, 3, padding=1)
        if gated_skips:
            self.skip_gates = nn.Parameter(torch.full((5,), -0.5))
    def gate(self, skip, idx):
        if not self.gated_skips:
            return skip
        return skip * torch.sigmoid(self.skip_gates[idx])
    def forward(self, bottleneck, skips, context, latent_map=None):
        x = torch.cat([bottleneck, latent_map], dim=1) if latent_map is not None else bottleneck
        s0, s1, s2, s3, s4 = skips
        x = self.u4(x); x = self.f4(torch.cat([x, self.gate(s4, 4)], dim=1)); x = self.film4(x, context)
        x = self.u3(x); x = self.f3(torch.cat([x, self.gate(s3, 3)], dim=1)); x = self.film3(x, context)
        x = self.u2(x); x = self.f2(torch.cat([x, self.gate(s2, 2)], dim=1)); x = self.film2(x, context)
        x = self.u1(x); x = self.f1(torch.cat([x, self.gate(s1, 1)], dim=1)); x = self.film1(x, context)
        x = self.u0(x); x = self.f0(torch.cat([x, self.gate(s0, 0)], dim=1)); x = self.film0(x, context)
        return self.out(x)

class PosteriorEncoder(nn.Module):
    def __init__(self, in_ch, width):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_ch, width), Down(width, width), Down(width, width * 2), Down(width * 2, width * 4), Down(width * 4, width * 8), Down(width * 8, width * 8)
        )
    def forward(self, condition, target):
        return self.net(torch.cat([condition, target], dim=1))

class GaussianHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.SiLU(inplace=True), nn.Linear(hidden_dim, hidden_dim), nn.SiLU(inplace=True))
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h).clamp(-12, 8)

class MultiLatentConditionalPriorResidualCVAE(nn.Module):
    def __init__(self, condition_ch, target_ch, meta_dim, latent_blocks, width=32, grid_n=256):
        super().__init__()
        reduced = grid_n // 32
        flat = width * 8 * reduced * reduced
        self.latent_blocks = dict(latent_blocks)
        self.latent_dim = sum(self.latent_blocks.values())
        self.width = width
        self.reduced = reduced
        self.condition_encoder = ConditionEncoder(condition_ch, width)
        self.posterior_encoder = PosteriorEncoder(condition_ch + target_ch, width)
        self.context_head = nn.Sequential(nn.Linear(width * 8 + meta_dim, width * 8), nn.SiLU(inplace=True), nn.Linear(width * 8, width * 8), nn.SiLU(inplace=True))
        pooled_dim = width * 8
        self.block_input_dims = {
            'global': pooled_dim + len(STATIC_IDX) + len(GLOBAL_META_IDX),
            'season': pooled_dim + len(SEASON_META_IDX),
            'weather': pooled_dim + len(WEATHER_IDX) + len(SEASON_META_IDX),
            'land': pooled_dim + len(LAND_IDX) + len(GLOBAL_META_IDX),
            'residual': pooled_dim + meta_dim,
        }
        post_shared_dim = width * 16
        self.post_shared = nn.Sequential(nn.Linear(flat + meta_dim, post_shared_dim), nn.SiLU(inplace=True), nn.Linear(post_shared_dim, post_shared_dim), nn.SiLU(inplace=True))
        self.prior_heads = nn.ModuleDict({
            name: GaussianHead(self.block_input_dims[name], width * 8, dim)
            for name, dim in self.latent_blocks.items()
        })
        self.posterior_heads = nn.ModuleDict({
            name: GaussianHead(post_shared_dim + self.block_input_dims[name], width * 8, dim)
            for name, dim in self.latent_blocks.items()
        })
        self.z_to_map = nn.Sequential(nn.Linear(self.latent_dim + width * 8, flat), nn.SiLU(inplace=True))
        self.base_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=0, gated_skips=True)
        self.residual_decoder = Decoder(width * 8, target_ch, width, context_dim=width * 8, latent_ch=width * 8, gated_skips=True)
    def channel_summary(self, condition, idxs):
        if not idxs:
            return condition.new_zeros((condition.shape[0], 0))
        return condition[:, idxs].mean(dim=(2, 3))
    def meta_select(self, meta, idxs):
        if not idxs:
            return meta.new_zeros((meta.shape[0], 0))
        return meta[:, idxs]
    def block_inputs(self, condition, bottleneck, meta):
        pooled = bottleneck.mean(dim=(2, 3))
        return {
            'global': torch.cat([pooled, self.channel_summary(condition, STATIC_IDX), self.meta_select(meta, GLOBAL_META_IDX)], dim=1),
            'season': torch.cat([pooled, self.meta_select(meta, SEASON_META_IDX)], dim=1),
            'weather': torch.cat([pooled, self.channel_summary(condition, WEATHER_IDX), self.meta_select(meta, SEASON_META_IDX)], dim=1),
            'land': torch.cat([pooled, self.channel_summary(condition, LAND_IDX), self.meta_select(meta, GLOBAL_META_IDX)], dim=1),
            'residual': torch.cat([pooled, meta], dim=1),
        }
    def context(self, bottleneck, meta):
        pooled = bottleneck.mean(dim=(2, 3))
        return self.context_head(torch.cat([pooled, meta], dim=1))
    def prior(self, condition, bottleneck, meta):
        inputs = self.block_inputs(condition, bottleneck, meta)
        return {name: self.prior_heads[name](inputs[name]) for name in self.latent_blocks}
    def posterior(self, condition, target, bottleneck, meta):
        post_feat = self.posterior_encoder(condition, target).flatten(1)
        shared = self.post_shared(torch.cat([post_feat, meta], dim=1))
        inputs = self.block_inputs(condition, bottleneck, meta)
        return {name: self.posterior_heads[name](torch.cat([shared, inputs[name]], dim=1)) for name in self.latent_blocks}
    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def sample_z(self, stats, use_mean=False):
        parts = []
        for name in self.latent_blocks:
            mu, logvar = stats[name]
            parts.append(mu if use_mean else self.reparameterize(mu, logvar))
        return torch.cat(parts, dim=1)
    def decode_from_z(self, condition, meta, z):
        b, skips = self.condition_encoder(condition)
        ctx = self.context(b, meta)
        zmap = self.z_to_map(torch.cat([z, ctx], dim=1)).view(condition.shape[0], self.width * 8, self.reduced, self.reduced)
        base = self.base_decoder(b, skips, ctx)
        residual = self.residual_decoder(b, skips, ctx, zmap)
        return base + residual, base, residual
    def decode(self, condition, meta, z=None, use_prior_mean=False):
        b, skips = self.condition_encoder(condition)
        ctx = self.context(b, meta)
        prior_stats = self.prior(condition, b, meta)
        if z is None:
            z = self.sample_z(prior_stats, use_mean=use_prior_mean)
        zmap = self.z_to_map(torch.cat([z, ctx], dim=1)).view(condition.shape[0], self.width * 8, self.reduced, self.reduced)
        base = self.base_decoder(b, skips, ctx)
        residual = self.residual_decoder(b, skips, ctx, zmap)
        return base + residual, base, residual, prior_stats
    def forward(self, condition, target, meta):
        b, skips = self.condition_encoder(condition)
        ctx = self.context(b, meta)
        prior_stats = self.prior(condition, b, meta)
        posterior_stats = self.posterior(condition, target, b, meta)
        z = self.sample_z(posterior_stats, use_mean=False)
        zmap = self.z_to_map(torch.cat([z, ctx], dim=1)).view(condition.shape[0], self.width * 8, self.reduced, self.reduced)
        base = self.base_decoder(b, skips, ctx)
        residual = self.residual_decoder(b, skips, ctx, zmap)
        return base + residual, base, residual, posterior_stats, prior_stats


def kl_normal_pair(mu_q, logvar_q, mu_p, logvar_p):
    return 0.5 * torch.mean(torch.sum(logvar_p - logvar_q + (logvar_q.exp() + (mu_q - mu_p).pow(2)) / logvar_p.exp().clamp_min(1e-8) - 1.0, dim=1))

def kl_blocks(posterior_stats, prior_stats):
    parts = {}
    total = None
    for name in LATENT_BLOCK_DIMS:
        mu_q, logvar_q = posterior_stats[name]
        mu_p, logvar_p = prior_stats[name]
        val = kl_normal_pair(mu_q, logvar_q, mu_p, logvar_p)
        parts[name] = val
        total = val if total is None else total + val
    return total, parts

def gradient_loss(pred, target, weight):
    mx = weight[:, :, :, 1:] * weight[:, :, :, :-1]
    my = weight[:, :, 1:, :] * weight[:, :, :-1, :]
    dxp = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dxt = target[:, :, :, 1:] - target[:, :, :, :-1]
    dyp = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    dyt = target[:, :, 1:, :] - target[:, :, :-1, :]
    return ((dxp - dxt).abs() * mx).sum() / mx.sum().clamp_min(1.0) + ((dyp - dyt).abs() * my).sum() / my.sum().clamp_min(1.0)

def laplacian(x):
    return -4*x[:, :, 1:-1, 1:-1] + x[:, :, :-2, 1:-1] + x[:, :, 2:, 1:-1] + x[:, :, 1:-1, :-2] + x[:, :, 1:-1, 2:]

def laplacian_loss(pred, target, weight):
    w = weight[:, :, 1:-1, 1:-1]
    return ((laplacian(pred) - laplacian(target)).abs() * w).sum() / w.sum().clamp_min(1.0)

def cvae_loss(pred, base, target, mask, weight, posterior_stats, prior_stats, beta, stage):
    w = (mask * weight).to(pred.dtype)
    if stage == 'base':
        recon = ((base - target).abs() * w).sum() / w.sum().clamp_min(1.0)
        grad = gradient_loss(base, target, w)
        lap = laplacian_loss(base, target, w)
        zero_parts = {name: pred.new_tensor(0.0) for name in LATENT_BLOCK_DIMS}
        return recon + LAMBDA_GRAD * grad + LAMBDA_LAP * lap, recon, pred.new_tensor(0.0), grad, lap, zero_parts
    recon = ((pred - target).abs() * w).sum() / w.sum().clamp_min(1.0)
    kl, kl_parts = kl_blocks(posterior_stats, prior_stats)
    grad = gradient_loss(pred, target, w)
    lap = laplacian_loss(pred, target, w)
    return recon + beta * kl + LAMBDA_GRAD * grad + LAMBDA_LAP * lap, recon, kl, grad, lap, kl_parts

grid_n = int(train_cache['target'].shape[-1])
model = MultiLatentConditionalPriorResidualCVAE(len(CONDITION_CHANNELS), len(TARGET_CHANNELS), len(META_CHANNELS), LATENT_BLOCK_DIMS, WIDTH, grid_n).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
print('model:', model.__class__.__name__)
print('params M:', sum(p.numel() for p in model.parameters()) / 1e6)


In [ ]:
# ### MODIFIED: MULTI-LATENT TRAINING LOOP v4.1
# block별 KL을 history/checkpoint_summary/checkpoint에 기록한다.

def run_epoch(loader, training, beta_eff, stage):
    model.train(training)
    totals = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0, 'grad': 0.0, 'lap': 0.0, 'n': 0}
    for name in LATENT_BLOCK_DIMS:
        totals[f'kl_{name}'] = 0.0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            condition = batch['condition'].to(DEVICE)
            target = batch['target'].to(DEVICE)
            mask = batch['target_mask'].to(DEVICE)
            weight = batch['target_weight'].to(DEVICE)
            meta = batch['meta'].to(DEVICE)
            pred, base, residual, posterior_stats, prior_stats = model(condition, target, meta)
            loss, recon, kl, grad, lap, kl_parts = cvae_loss(pred, base, target, mask, weight, posterior_stats, prior_stats, beta_eff, stage)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            bs = condition.shape[0]
            for k, v in [('loss', loss), ('recon', recon), ('kl', kl), ('grad', grad), ('lap', lap)]:
                totals[k] += float(v.detach().cpu()) * bs
            for name, val in kl_parts.items():
                totals[f'kl_{name}'] += float(val.detach().cpu()) * bs
            totals['n'] += bs
    n = max(totals['n'], 1)
    keys = ['loss', 'recon', 'kl', 'grad', 'lap'] + [f'kl_{name}' for name in LATENT_BLOCK_DIMS]
    return {k: totals[k] / n for k in keys}

history = []
ckpt_rows = []
best_val = float('inf')
best_balanced = float('inf')
bad = 0
latest_path = RUN_DIR / 'latest.pt'
best_path = RUN_DIR / 'best_val.pt'
best_balanced_path = RUN_DIR / 'best_balanced.pt'
checkpoint_dir = RUN_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    stage = 'base' if epoch <= STAGE1_BASE_EPOCHS else 'cvae'
    beta_eff = 0.0 if stage == 'base' else BETA * min(1.0, (epoch - STAGE1_BASE_EPOCHS) / KL_WARMUP_EPOCHS)
    tr = run_epoch(train_loader, True, beta_eff, stage)
    va = run_epoch(val_loader, False, beta_eff, stage) if val_loader is not None else tr
    scheduler.step(va['loss'])
    row = {'epoch': epoch, 'stage': stage, 'beta_eff': beta_eff, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}, 'lr': optimizer.param_groups[0]['lr']}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False, encoding='utf-8-sig')
    ckpt = {
        'epoch': epoch,
        'stage': stage,
        'model_name': model.__class__.__name__,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'normalization_stats': stats,
        'condition_channels': CONDITION_CHANNELS,
        'target_channels': TARGET_CHANNELS,
        'meta_channels': META_CHANNELS,
        'latent_blocks': LATENT_BLOCK_DIMS,
        'latent_dim': LATENT_DIM,
        'width': WIDTH,
        'beta': BETA,
        'kl_warmup_epochs': KL_WARMUP_EPOCHS,
        'stage1_base_epochs': STAGE1_BASE_EPOCHS,
        'grid_n': grid_n,
        'metrics': row,
    }
    torch.save(ckpt, latest_path)
    if SAVE_EVERY_EPOCH:
        torch.save(ckpt, checkpoint_dir / f'epoch_{epoch:03d}.pt')
    saved = ['latest']
    if va['loss'] < best_val:
        best_val = va['loss']
        bad = 0
        torch.save(ckpt, best_path)
        saved.append('best_val')
    else:
        bad += 1
    if stage == 'cvae' and va['kl'] >= BALANCED_MIN_KL and va['loss'] < best_balanced:
        best_balanced = va['loss']
        torch.save(ckpt, best_balanced_path)
        saved.append('best_balanced')
    ckpt_row = {'epoch': epoch, 'stage': stage, 'saved': ','.join(saved), 'val_loss': va['loss'], 'val_kl': va['kl'], 'train_loss': tr['loss'], 'train_kl': tr['kl']}
    for name in LATENT_BLOCK_DIMS:
        ckpt_row[f'val_kl_{name}'] = va[f'kl_{name}']
        ckpt_row[f'train_kl_{name}'] = tr[f'kl_{name}']
    ckpt_rows.append(ckpt_row)
    pd.DataFrame(ckpt_rows).to_csv(RUN_DIR / 'checkpoint_summary.csv', index=False, encoding='utf-8-sig')
    kl_msg = ' '.join([f"{name}={va[f'kl_{name}']:.4f}" for name in LATENT_BLOCK_DIMS])
    print(f"{epoch:03d} {stage} beta={beta_eff:.2e} train={tr['loss']:.5f} recon={tr['recon']:.5f} kl={tr['kl']:.5f} val={va['loss']:.5f} val_kl={va['kl']:.5f} [{kl_msg}] lr={optimizer.param_groups[0]['lr']:.2e} saved={','.join(saved)}")
    if bad >= PATIENCE:
        print('early stop')
        break
print('best_val:', best_path)
print('best_balanced:', best_balanced_path if best_balanced_path.exists() else 'not saved')


In [ ]:
# ### MODIFIED: MULTI-LATENT POSTERIOR EVALUATION v4.1
@torch.no_grad()
def evaluate(paths, loader, split_name, checkpoint_path=best_path):
    if loader is None:
        return pd.DataFrame()
    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    model.eval()
    pred_dir = RUN_DIR / 'predictions' / split_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    sample_idx = 0
    for batch in loader:
        condition = batch['condition'].to(DEVICE)
        target = batch['target'].to(DEVICE)
        mask = batch['target_mask'].to(DEVICE)
        meta = batch['meta'].to(DEVICE)
        pred, base, residual, posterior_stats, prior_stats = model(condition, target, meta)
        pred_np = pred.detach().cpu().numpy()
        base_np = base.detach().cpu().numpy()
        target_np = target.detach().cpu().numpy()
        mask_np = mask.detach().cpu().numpy().astype(bool)
        clear_np = batch['clear_mask'].numpy().astype(bool)
        cloud_np = batch['cloud_fill_mask'].numpy().astype(bool)
        for b in range(pred_np.shape[0]):
            p = paths[sample_idx]
            pred_den = denormalize(pred_np[b], stats['target'], TARGET_CHANNELS)
            base_den = denormalize(base_np[b], stats['target'], TARGET_CHANNELS)
            target_den = denormalize(target_np[b], stats['target'], TARGET_CHANNELS)
            diff = pred_den[0] - target_den[0]
            for region, rmask in [('all', mask_np[b, 0]), ('landsat_clear', clear_np[b, 0] & mask_np[b, 0]), ('cloud_filled', cloud_np[b, 0] & mask_np[b, 0])]:
                valid = rmask & np.isfinite(diff)
                if valid.any():
                    rows.append({'split': split_name, 'file': p.name, 'region': region, 'rmse_k': float(np.sqrt(np.mean(diff[valid]**2))), 'mae_k': float(np.mean(np.abs(diff[valid]))), 'bias_k': float(np.mean(diff[valid])), 'valid_fraction': float(valid.mean())})
            np.savez_compressed(pred_dir / p.name, prediction=pred_den, base=base_den, target=target_den, target_mask=mask_np[b], clear_mask=clear_np[b, 0], cloud_fill_mask=cloud_np[b, 0], target_channels=np.array(TARGET_CHANNELS))
            sample_idx += 1
    return pd.DataFrame(rows)

metrics = pd.concat([evaluate(val_paths, val_loader, 'val'), evaluate(test_paths, test_loader, 'test')], ignore_index=True)
metrics.to_csv(RUN_DIR / 'metrics_detail.csv', index=False, encoding='utf-8-sig')
summary = metrics.groupby(['split', 'region'])[['rmse_k', 'mae_k', 'bias_k', 'valid_fraction']].mean().reset_index()
summary.to_csv(RUN_DIR / 'metrics_summary.csv', index=False, encoding='utf-8-sig')
summary


In [ ]:
# ### MODIFIED: MULTI-LATENT PRIOR EVALUATION v4.1
@torch.no_grad()
def evaluate_prior(paths, loader, split_name, mode='prior_mean', n_samples=8, checkpoint_path=best_path):
    if loader is None:
        return pd.DataFrame()
    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    model.eval()
    rows = []
    pred_dir = RUN_DIR / 'predictions_prior' / mode / split_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    sample_idx = 0
    for batch in loader:
        condition = batch['condition'].to(DEVICE)
        target = batch['target'].to(DEVICE)
        mask = batch['target_mask'].to(DEVICE)
        meta = batch['meta'].to(DEVICE)
        if mode == 'prior_mean':
            pred, base, residual, prior_stats = model.decode(condition, meta, use_prior_mean=True)
        else:
            preds = [model.decode(condition, meta, use_prior_mean=False)[0] for _ in range(n_samples)]
            pred = torch.stack(preds, dim=0).mean(dim=0)
        pred_np = pred.detach().cpu().numpy()
        target_np = target.detach().cpu().numpy()
        mask_np = mask.detach().cpu().numpy().astype(bool)
        clear_np = batch['clear_mask'].numpy().astype(bool)
        cloud_np = batch['cloud_fill_mask'].numpy().astype(bool)
        for b in range(pred_np.shape[0]):
            p = paths[sample_idx]
            pred_den = denormalize(pred_np[b], stats['target'], TARGET_CHANNELS)
            target_den = denormalize(target_np[b], stats['target'], TARGET_CHANNELS)
            diff = pred_den[0] - target_den[0]
            for region, rmask in [('all', mask_np[b, 0]), ('landsat_clear', clear_np[b, 0] & mask_np[b, 0]), ('cloud_filled', cloud_np[b, 0] & mask_np[b, 0])]:
                valid = rmask & np.isfinite(diff)
                if valid.any():
                    rows.append({'split': split_name, 'mode': mode, 'file': p.name, 'region': region, 'rmse_k': float(np.sqrt(np.mean(diff[valid]**2))), 'mae_k': float(np.mean(np.abs(diff[valid]))), 'bias_k': float(np.mean(diff[valid])), 'valid_fraction': float(valid.mean())})
            np.savez_compressed(pred_dir / p.name, prediction=pred_den, target=target_den, target_mask=mask_np[b], clear_mask=clear_np[b, 0], cloud_fill_mask=cloud_np[b, 0], target_channels=np.array(TARGET_CHANNELS), mode=np.array(mode))
            sample_idx += 1
    return pd.DataFrame(rows)

prior_metrics = pd.concat([
    evaluate_prior(val_paths, val_loader, 'val', 'prior_mean'),
    evaluate_prior(test_paths, test_loader, 'test', 'prior_mean'),
    evaluate_prior(val_paths, val_loader, 'val', 'prior_ensemble', n_samples=8),
    evaluate_prior(test_paths, test_loader, 'test', 'prior_ensemble', n_samples=8),
], ignore_index=True)
prior_metrics.to_csv(RUN_DIR / 'metrics_prior_detail.csv', index=False, encoding='utf-8-sig')
prior_summary = prior_metrics.groupby(['split', 'mode', 'region'])[['rmse_k', 'mae_k', 'bias_k', 'valid_fraction']].mean().reset_index()
prior_summary.to_csv(RUN_DIR / 'metrics_prior_summary.csv', index=False, encoding='utf-8-sig')
prior_summary


In [ ]:
# Drive 백업 셀: 학습 후 반드시 실행
# local SSD의 checkpoint/results를 Drive로 복사하고, 로컬로 가져올 pt 경로를 출력한다.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

from pathlib import Path
import json
import shutil

src = Path(os.environ.get('ATTEMPT4_RUN_DIR', '/content/attempt4_runs'))
dst = Path('/content/drive/MyDrive/attempt4_cvae_runs')
if not src.exists():
    raise FileNotFoundError(src)
dst.mkdir(parents=True, exist_ok=True)

# Python copy로 Colab 기본 환경에서도 동작하게 둔다. 기존 파일은 덮어쓴다.
for item in src.rglob('*'):
    rel = item.relative_to(src)
    out = dst / rel
    if item.is_dir():
        out.mkdir(parents=True, exist_ok=True)
    else:
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, out)

run_drive = dst / 'runs' / 'lst_cvae_condprior_residual_v4'
pt_paths = {
    'latest': run_drive / 'latest.pt',
    'best_val': run_drive / 'best_val.pt',
    'best_balanced': run_drive / 'best_balanced.pt',
}
existing = {k: str(v) for k, v in pt_paths.items() if v.exists()}
all_epoch_pts = sorted((run_drive / 'checkpoints').glob('epoch_*.pt')) if (run_drive / 'checkpoints').exists() else []
if all_epoch_pts:
    existing['last_epoch_checkpoint'] = str(all_epoch_pts[-1])
    existing['checkpoint_dir'] = str(run_drive / 'checkpoints')

manifest_path = run_drive / 'checkpoint_paths.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(existing, ensure_ascii=False, indent=2), encoding='utf-8')

print('backup done:', src, '->', dst)
print('checkpoint path manifest:', manifest_path)
for name, path in existing.items():
    print(f'{name}: {path}')
